# 🛠️ 大模型工具调用之 Function Calling 实战

在前面的检索增强生成 (RAG) 实验中，大模型只是通过**被动读取**我们预先加载好的检索片段来回答问题。这种方式下，大模型就像一个闭门读书的学者。

本实验将探讨如何让大模型成为一个**主动与外部世界交互的“智能体” (Agent)**。
我们将具体学习两个模块：
- **最原始的标签式工具调用原理**：使用特定的标签与 Python 的 `eval()` 实现最直观、手动的工具触发与执行；
- **自定义 Tools 以及标准的 Function Calling**：学习大模型行业标准的 Function Calling 协议，理解大模型如何自动生成结构化 JSON 参数，彻底摆脱 `eval()` 的安全隐患。

## 🛠️ 环境初始化：API 密钥与 OpenAI 客户端配置

为了运行本 Notebook，我们需要配置大模型 API。请在下方填入您的 API Key 和 Base URL。本代码默认支持通义千问 (DashScope) 与硅基流动 (SiliconFlow)，也可以直接使用 OpenAI 或其他兼容的 API 接口。

In [ ]:
import os
import json
import time
import httpx
from openai import OpenAI

# ============================================================
# 👇 请在下方引号内填入您的 API Key（也可以直接读取系统环境变量）
# ============================================================
API_KEY = ""         # 例如: "sk-abc123..."
BASE_URL = "https://api.siliconflow.cn/v1"        # 例如: "https://api.siliconflow.cn/v1" 或 "https://dashscope.aliyuncs.com/compatible-mode/v1"
MODEL_NAME = "deepseek-ai/DeepSeek-V3"      # 例如: "deepseek-ai/DeepSeek-V3" 或 "qwen-plus"

# 优先从上面填写的变量读取，其次读取系统环境变量
api_key = API_KEY or os.environ.get("OPENAI_API_KEY") or os.environ.get("SILICONFLOW_API_KEY") or os.environ.get("DASH_SCOPE_API_KEY")
base_url = BASE_URL or os.environ.get("OPENAI_API_BASE")
model_name = MODEL_NAME or os.environ.get("OPENAI_API_MODEL")

# 自动识别常见服务商的环境变量
if not base_url:
    if os.environ.get("SILICONFLOW_API_KEY"):
        base_url = "https://api.siliconflow.cn/v1"
        model_name = model_name or "deepseek-ai/DeepSeek-V3"
    elif os.environ.get("DASH_SCOPE_API_KEY"):
        base_url = "https://dashscope.aliyuncs.com/compatible-mode/v1"
        model_name = model_name or "qwen-plus"
    else:
        base_url = "https://api.openai.com/v1"
        model_name = model_name or "gpt-4o-mini"

if not api_key:
    raise ValueError("❌ 未检测到 API Key！请在上方单元格中配置您的 API Key 或设置系统环境变量。")

# 💡 如果您在 Windows 环境下使用 VPN 或代理工具，可能会遇到 SSL/ConnectError 报错。
# 此时可以尝试取消下方这行代码的注释，使用禁用了 SSL 证书验证的自定义 httpx 客户端：
# openai_client = OpenAI(api_key=api_key, base_url=base_url, http_client=httpx.Client(verify=False))
openai_client = OpenAI(api_key=api_key, base_url=base_url)
print(f"✨ 客户端实例化成功！当前使用的接口地址为: {base_url}，模型为: {model_name}")

## 📂 测试数据生成

为了进行文件查找实验，我们需要在本地创建一个测试目录 `data/fc_test`，并在其中放入几个测试文件，包括我们要让大模型查找的目标文件 `target_simple.txt`。

In [ ]:
import shutil

# 创建测试目录
test_dir = "data/fc_test"
if os.path.exists(test_dir):
    shutil.rmtree(test_dir)
os.makedirs(test_dir, exist_ok=True)

# 写入一些干扰文件与目标文件
files_to_create = {
    "report.txt": "2026年年度财务报告：公司业绩稳步上升。",
    "config.json": '{"port": 8080, "debug": false}',
    "target_simple.txt": "恭喜你！成功找到了 target_simple.txt 文件。密钥为: SIMPLE_SUCCESS_2026",
    "notes.log": "2026-06-03 12:00:00 - Server started successfully."
}

for filename, content in files_to_create.items():
    with open(os.path.join(test_dir, filename), "w", encoding="utf-8") as f:
        f.write(content)

print(f"💾 测试目录 {test_dir} 准备完毕，已创建 {len(files_to_create)} 个文件。")

---
## 🧱 用标签与 eval 实现最原始的工具调用（原理演示）

在 OpenAI 官方推出标准的 Function Calling 接口之前，早期的 AI 智能体（Agent）是通过在 Prompt 中约定标签格式来调用工具的。

它的核心思想是：
1. **约定格式**：在 Prompt 中规定，如果需要使用工具，必须输出特定格式的代码标签，例如 `<tool>get_temperature('北京', '10月1日')</tool>`。
2. **提取与执行**：后台程序解析大模型的输出，提取出标签包裹的 Python 函数表达式，并用 `eval()` 动态执行。
3. **反馈给模型**：把执行结果作为输入包装成 `<tool_output>...</tool_output>` 再反馈给大模型，直到大模型给出最终自然语言答案。

> ⚠️ **安全警告**：在实际工程中，直接使用 `eval()` 运行大模型输出的不受控代码是**极其危险的**（存在代码注入漏洞）。本模块仅用于教学原理解释。

In [ ]:
# 1. 定义可供大模型调用的本地函数工具
def multiply(a, b):
    """返回 a 乘以 b"""
    return a * b

def divide(a, b):
    """返回 a 除以 b"""
    return a / b

def get_temperature(city: str, time: str) -> str:
    """返回 city 在 time 的气温"""
    # 模拟天气数据返回
    return f"{city}在{time}的温度是 22 度，秋高气爽。"

# 2. 系统 Prompt (System Prompt) 引导大模型如何输出工具调用标签
tool_instruction = """有必要可以使用工具，每一个工具都是函数。
使用工具的方式为输出 "<tool>[使用工具指令]</tool>"。
你会得到返回结果 "<tool_output>[工具返回的结果]</tool_output>"。
如果有使用工具的话，你应该告诉用户工具返回的结果。

可用工具：
multiply(a,b): 返回 a 乘以 b
divide(a,b): 返回 a 除以 b
get_temperature(city,time): 返回 city 在 time 的气温，注意 city 和 time 都是string类型"""

# 定义用户问题
user_input = "告诉我北京10月1日天气如何啊？"
# 也可以测试数学计算问题：
# user_input = "请计算 111 * 222 / 777 是多少？"

messages = [
    {"role": "system", "content": tool_instruction},
    {"role": "user", "content": user_input}
]

# 3. ReAct 循环
step = 0
while True:
    print(f"\n--- 🔄 第 {step+1} 轮大模型推理中... ---")
    
    response = openai_client.chat.completions.create(
        model=model_name,
        messages=messages,
        temperature=0.1
    )
    
    response_text = response.choices[0].message.content
    
    # 检测大模型是否输出工具调用标签
    if "</tool>" in response_text:
        start_idx = response_text.find("<tool>") + len("<tool>")
        end_idx = response_text.find("</tool>")
        command = response_text[start_idx:end_idx].strip()
        
        print("🤖 大模型原始响应: ", response_text)
        print("⚙️ 提取出待调用的工具指令: ", command)
        
        # 使用 eval 动态执行本地函数
        try:
            tool_output = str(eval(command))
            print("💾 本地工具函数 eval 执行结果: ", tool_output)
        except Exception as e:
            tool_output = f"工具执行报错: {str(e)}"
            print("❌ 工具执行报错: ", e)
            
        # 裁剪模型回复：只保留到 </tool> 的部分
        clean_response = response_text.split("</tool>")[0] + "</tool>"
        
        messages.append({"role": "assistant", "content": clean_response})
        
        # 反馈执行结果
        user_feedback = f"<tool_output>{tool_output}</tool_output>"
        messages.append({"role": "user", "content": user_feedback})
        
    else:
        # 没有工具标签，输出最终回复
        print("🏆 大模型最终自然语言答复:")
        print("=" * 65)
        print(response_text)
        print("=" * 65)
        break
        
    step += 1
    if step > 5:
        print("⚠️ 思考轮数达到上限，防止死循环。")
        break

---
## 🧱 自定义 Tools 以及标准的 Function Calling

### 💡 为什么需要工业标准的 Function Calling？
模块零手工解析文本标签有几个痛点：格式容易生成错误、`eval()` 执行有严重安全隐患。

因此，大模型服务商定义了标准的 **Function Calling 协议**。大模型接收 `tools` 描述（JSON Schema 格式），在需要时返回一个结构化的 `tool_calls` 参数对象，大模型只做**参数提取与决策**，具体执行由后台代码安全地调用真实函数，彻底避免了 `eval` 的注入危险。

In [ ]:
def list_top_level_files() -> list:
    """列出 data/fc_test 目录下的所有文件名。"""
    base_dir = "data/fc_test"
    if not os.path.exists(base_dir):
        return []
    return [f for f in os.listdir(base_dir) if os.path.isfile(os.path.join(base_dir, f))]

def check_file_exists(filename: str) -> str:
    """在本地 data/fc_test 目录下检查指定的文件是否存在，如果存在则读取其内容并返回。"""
    base_dir = "data/fc_test"
    filepath = os.path.join(base_dir, filename)
    if os.path.exists(filepath):
        with open(filepath, 'r', encoding='utf-8') as f:
            return f.read()
    return f"文件 {filename} 不存在于 {base_dir} 中。"

print("✅ 基础文件工具函数就绪。")

In [ ]:
# 1. 定义大模型能理解的 Tool Schema
my_tools = [
    {
        "type": "function",
        "function": {
            "name": "list_top_level_files",
            "description": "列出本地测试目录 data/fc_test 下的所有文件名。",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_file_exists",
            "description": "在本地 data/fc_test 目录下检查指定的文件是否存在，如果存在则读取其内容并返回。",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {
                        "type": "string",
                        "description": "要检查的文件名，例如 'target_simple.txt'"
                    }
                },
                "required": ["filename"]
            }
        }
    }
]

print("✅ 成功定义自定义 Tool Schema。")

In [ ]:
# 2. 标准的 Function Calling 执行流程
user_query = "帮我查找 data/fc_test 目录下是否存在名为 target_simple.txt 的文件，如果存在请告诉我其内容。"
messages = [{"role": "user", "content": user_query}]

print("📤 发送用户提问，让大模型决策并生成 Tool Call 参数...")
response = openai_client.chat.completions.create(
    model=model_name,
    messages=messages,
    tools=my_tools,
    tool_choice="auto"
)

assistant_message = response.choices[0].message
tool_calls = assistant_message.tool_calls

messages.append(assistant_message)

if tool_calls:
    print("🤖 大模型决策结果: 【需要调用外部工具】")
    for tool_call in tool_calls:
        func_name = tool_call.function.name
        func_args = json.loads(tool_call.function.arguments)
        
        print(f"   - 待执行工具: {func_name} | 参数: {func_args}")
        
        # 根据大模型输出安全地调用真实 Python 函数
        if func_name == "list_top_level_files":
            local_result = str(list_top_level_files())
        elif func_name == "check_file_exists":
            local_result = check_file_exists(func_args.get("filename"))
        else:
            local_result = f"未知工具: {func_name}"
            
        print(f"   ▶️ 真实函数执行结果为: '{local_result}'")
        
        # 将执行结果作为 tool 消息追加进对话历史
        messages.append({
            "role": "tool",
            "tool_call_id": tool_call.id,
            "name": func_name,
            "content": local_result
        })
        
    # 发送完整的对话历史给大模型，获取最终答案
    print("\n📤 正在将本地结果追加回上下文，请求大模型生成最终总结...")
    final_response = openai_client.chat.completions.create(
        model=model_name,
        messages=messages
    )
    final_answer = final_response.choices[0].message.content
    
    print("\n🤖 大模型最终总结回答:")
    print("=" * 60)
    print(final_answer)
    print("=" * 60)
else:
    print("无需执行工具，大模型直接答复:")
    print(assistant_message.content)

## 📈 课后知识复盘与思考练习

### 🧠 思考题
在模块零的标签调用中，我们使用了 Python 的 `eval()`。请举例说明，如果用户不怀好意地设计输入，例如通过提问诱导大模型输出包含 `__import__('os').system('rm -rf /')` 标签的文本，后台的 `eval()` 执行会带来什么危害？标准的 Function Calling 又是如何彻底避免这个安全风险的？